# Deep Learning for System Identification
## Introduction: Least-squares estimation and simulation error minimization
### Authors: Dario Piga, Marco Forgione, IDSIA SUPSI/USI, Lugano, Switzerland

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

### Load data from the [Cascaded Tanks benchmark](https://www.nonlinearbenchmark.org/benchmarks/cascaded-tanks)

In [ ]:
dataset = pd.read_csv(
    os.path.join("..", "..", "data", "CascadedTanks", "dataBenchmark.csv")
)  # change path according to the location of your csv file

In [ ]:
dataset.head()

In [ ]:
# rename the columns in training and test sequences (we typically call validation set the one used to tune algorithms' hyper-parameters)
dataset = dataset.rename(
    columns={
        "uEst": "u_train",
        "uVal": "u_test",
        "yEst": "y_train",
        "yVal": "y_test",
    }
)

dataset.head()

### Dataset visualization

In [ ]:
plt.figure(figsize=(12, 10))
plt.suptitle("Cascaded Tanks Dataset")
cols, rows = 2, 2

sequence_names = ("u_train", "y_train", "u_test", "y_test")

for i, n in enumerate(sequence_names):
    plt.subplot(cols, rows, i + 1)
    plt.plot(dataset[n])
    plt.title(sequence_names[i])
    plt.xlabel("Time step (-)")
    plt.ylabel("Magnitude (V)")

### Normalize data (good practice)

In [ ]:
u_train = np.array(dataset["u_train"])
y_train = np.array(dataset["y_train"])

u_test = np.array(dataset["u_test"])
y_test = np.array(dataset["y_test"])
u_train.shape, y_train.shape

In [ ]:
# compute statistics on the training set
u_mean = np.mean(u_train)
u_std = np.std(u_train)
y_mean = np.mean(y_train)
y_std = np.std(y_train)

# normalize the training and test sets with the training set statistics
u_train = (u_train - u_mean) / u_std
u_test = (u_test - u_mean) / u_std

y_train = (y_train - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

print(f"Training input: mean = {np.mean(u_train):.2f}; std = {np.std(u_train):.2f}")
print(f"Training output: mean = {np.mean(y_train):.2f}; std = {np.std(y_train):.2f}")

print(f"Test input: mean = {np.mean(u_test):.2f}; std = {np.std(u_test):.2f}")
print(f"Test output: mean = {np.mean(y_test):.2f}; std = {np.std(y_test):.2f}")

## Part I: Linear regression



We would like to fit a linear *AutoRegressive eXogenous* (ARX) model with structure:
$$
M({\theta}) : 
\hat y(k) = \overbrace{\begin{bmatrix}  y(k-1)& \dots& y(k-n_a)& u(k)& \dots & u(k-n_b)  \end{bmatrix}}^{=\phi^\top(k)} {\theta}.
$$

Here, the vector $\theta$ contains the $n_a + n_b + 1$ ARX model coefficients:

$$ \theta = \begin{bmatrix}a_1 & a_1 & \dots & a_{n_a} &  b_0 & b_1 & \dots & b_{n_b}\end{bmatrix}^\top.$$

The extension to a NARX structure where the regressor $\phi(k)$ contains non-linear terms of $u(k), y(k)$ is trivial.

In [ ]:
# create regressor
na = 2  # output lag
nb = 2  # input lags

### Create regressor

We first need to build the regression matrix $\Phi$ containing in each row the regressor vectors $\phi(k)$:

\begin{equation*}
\overbrace{
\begin{bmatrix}
\hat y(1) \\
\vdots \\
\hat y(k)\\
\vdots \\
\hat y(N)) \\
\end{bmatrix}
}^{=\hat Y}= 
\overbrace{
\begin{bmatrix}
y(0) & \dots  & y(1-n_a) & u(1) & \dots & u(1 - n_b)\\
\vdots \\
y(k-1) & \dots &y(k-n_a) & u(k) & \dots & u(k-n_b)\\
\vdots \\
y(N-1) & \dots  & y(N-n_a) & u(N) & \dots & u(N - n_b)\\
\end{bmatrix}
}^{=\Phi}
\theta
\end{equation*}

We actually skip the first $n = \max(n_a, n_b)$ rows, as the regressor is not defined there...

In [ ]:
n = max((na, nb))  # number of initial conditions

N_train = np.shape(y_train)[0]  # number of samples
phi_train = np.zeros((N_train - n, na + nb + 1))
for idx in range(n, N_train):
    y_lag = np.flip(y_train[idx - na : idx]) # y(k-1), ..., y(k-na), 
    u_lag = np.flip(u_train[idx - nb : idx + 1]) # u(k), ..., u(k-nb)
    phi_train[idx - n, :] = np.concatenate((y_lag, u_lag))

N_test = np.shape(y_test)[0]  # number of samples
phi_test = np.zeros((N_test - n, na + nb + 1))
for idx in range(n, N_test):
    y_lag = np.flip(y_test[idx - na : idx]) # y(k-1), ..., y(k-na), 
    u_lag = np.flip(u_test[idx - nb : idx + 1]) # u(k), ..., u(k-nb)
    phi_test[idx - n, :] = np.concatenate((y_lag, u_lag))

# The regressors are associated to the output from time n to time N-1
y_train_ls = np.reshape(y_train[n:], (-1, 1))
y_test_ls = np.reshape(y_test[n:], (-1, 1))

In [ ]:
phi_train.shape, y_train_ls.shape

### Least-squares estimate

We obtain the least squares estimate by solving:

$$\theta_{LS} = (\Phi^\top \Phi)^{-1}\Phi^\top Y $$

In [ ]:
theta_ls = np.linalg.solve(
    np.matmul(phi_train.transpose(), phi_train),  # Phi_train^T * Phi_train
    np.matmul(phi_train.transpose(), y_train_ls),  # Phi_train^T * Y
)
print(f"Least-square estimate: \n {theta_ls}")
# theta_ls_ = np.linalg.lstsq(phi_train, y_train_ls, rcond=None)[0]# alternative (better) way to compute the least-square estimate

# Training using scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

lin_model = LinearRegression(fit_intercept=False)  # no intercept
lin_model.fit(phi_train, y_train_ls)

# comparison of theta coefficients with results of LS:
print(f"Scikit-learn coefficients: \n {lin_model.coef_}")
print(f"Explicit Least squares coefficients: \n {theta_ls.transpose()}")

# Testing one-step prediction

In [ ]:
def regression_metrics(y_true, y_model):
    y_true = y_true.ravel() # flatten to avoid ambiguity in shape
    y_model = y_model.ravel()
    err = y_true - y_model
    MSE = np.mean(err**2)
    R2 = 1 - (MSE / np.mean((y_true - np.mean(y_true)) ** 2))
    return MSE, R2

In [ ]:
y_train_hat = np.matmul(phi_train, theta_ls) # phi_train @ theta_ls
mse_train, R2_train = regression_metrics(y_train_ls, y_train_hat)
print("Training performance:")
print(f"MSE: {mse_train:.4f}; R2: {R2_train:.4f}")

y_test_hat = np.matmul(phi_test, theta_ls)
mse_test, R2_test = regression_metrics(y_test_ls, y_test_hat)
print("Test performance:")
print(f"MSE: {mse_test:.4f}; R2: {R2_test:.4f}")


plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(y_train_ls * y_std + y_mean, label="True")
plt.plot(y_train_hat * y_std + y_mean, label="Estimated")
plt.title("Training")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(y_test_ls * y_std + y_mean, label="True")
plt.plot(y_test_hat * y_std + y_mean, label="Estimated")
plt.title("Test")
plt.legend()
plt.show()

### Question: Performance seems to be great! Do you think we obtained a good model?

# Testing open-loop simulation

In [ ]:
# simulation performance
def simulate(theta, u, na, nb, y0=None):

    n = np.max((na, nb))
    N = u.shape[0]
    y_sim = np.zeros(N)
    if y0 is not None:
        y_sim[0:n] = y0

    for idx in range(n, N):
        # the regressor vector is built on the fly based on the previous *simulated* values
        y_lag = np.flip(y_sim[idx - na : idx])  # \hat y(k-1), ..., \hat y(k-na) 
        u_lag = np.flip(u[idx - nb : idx + 1])  # u(k), u(k-1), ..., u(k-nb)
        phi_sim = np.concatenate((y_lag, u_lag))
        y_sim[idx] = np.matmul(phi_sim, theta)[0]

    return y_sim

In [ ]:
y_train_sim = simulate(theta_ls, u_train, na, nb)
mse_train, R2_train = regression_metrics(y_train[n:], y_train_sim[n:])
print(f"Train - MSE: {mse_train:.4f}, R2: {R2_train:.4f}")

y_test_sim = simulate(theta_ls, u_test, na, nb, y0=np.array([0, 0]))
mse_test, R2_test = regression_metrics(y_test[n:], y_test_sim[n:])
print(f"Test - MSE: {mse_test:.4f}, R2: {R2_test:.4f}")

plt.figure()
plt.plot(y_test * y_std + y_mean, label="True"),
plt.plot(y_test_sim[n:] * y_std + y_mean, label="Simulated")
plt.title("Test data")
plt.legend()
plt.show()

## Part II: Simulation error minimization (Implementation based on PyTorch)

In [ ]:
import torch
torch.random.manual_seed(0)


# Simulation error minimization as a differentiable loss function
def simulation_loss(theta, u, y, na, nb):

    n = np.max((na, nb))
    seq_len = u.shape[0]
    y_sim = torch.zeros(seq_len, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    u = torch.tensor(u, dtype=torch.float32)

    for idx in range(n, seq_len):

        # build the regressor with previously *simulated* outputs and inputs
        y_lag = torch.flip(y_sim[idx - na : idx], dims=[0]) # \hat y(k-1), ..., \hat y(k-na) 
        u_lag = torch.flip(u[idx - nb : idx + 1], dims=[0]) # u(k), u(k-1), ..., u(k-nb)
        phi_sim = torch.cat((y_lag, u_lag))

        y_sim[idx] = torch.matmul(phi_sim, theta)

    loss = torch.mean((y[n:] - y_sim[n:]) ** 2)

    return loss

In [ ]:
# initialize the parameters to the least-square solution
theta = torch.tensor(theta_ls, dtype=torch.float32, requires_grad=True)  

In [ ]:
loss = simulation_loss(theta, u_train, y_train, na, nb)
loss

In [ ]:
loss.backward() # run automatic differentiation...

In [ ]:
theta.grad # gradients wrt theta are computed!

In [ ]:
# iterative gradient-based optimization of the simulation loss

In [ ]:
#max_iter = 10
max_iter = 200
lr = 1e-3

optimizer = torch.optim.Adam([theta], lr=lr)

for train_iter in range(max_iter):

    optimizer.zero_grad()

    # forward pass
    loss = simulation_loss(theta, u_train, y_train, na, nb)

    # backward pass (derivative computation)
    loss.backward()

    optimizer.step()

    if train_iter % 10 == 0:
        print(f"Iteration: {train_iter}, loss: {loss:2f}")

print(f"Estimated parameters: {theta}")

In [ ]:
y_train_sim = simulate(theta.detach().numpy(), u_train, na, nb, y0=np.array([0, 0]))
mse_test, R2_test = regression_metrics(y_train[n:], y_train_sim[n:])
print(f"Train - MSE: {mse_train:.4f}, R2: {R2_train:.4f}")


y_test_sim = simulate(theta.detach().numpy(), u_test, na, nb, y0=np.array([0, 0]))
mse_test, R2_test = regression_metrics(y_test[n:], y_test_sim[n:])
print(f"Test - MSE: {mse_test:.4f}, R2: {R2_test:.4f}")

plt.figure()
plt.plot(y_test),
plt.plot(y_test_sim[n:])
plt.show()

# Part III: L1 Regularization


We deal with a NFIR model structure with a high-dimensional regressor:

$$
M({\theta}) : 
\hat y(k) = F^\top \bigg (u(k), u(k-1), \dots  u(k-n_b) \bigg) {\theta}
$$

- long memory ($n_b$ large)
- nonlinear transformations of past inputs (polynomials)


We apply **L1 Regularization** to perform **automatic selection** of a relevant subset of the regressor terms.

Discuss the impact of the **regularization parameter** on:
- model sparsity (i.e., number of non-zero coefficients),  
- and overall predictive performance.



### Build the regressor matrix

In [ ]:
nfir = 300  # FIR lags (=nb for NARX)
max_power = 1 # linear FIR


# Each y and u term is expanded to [x, x^2, x^3]
reg_dim = (nfir + 1) * max_power  # original regressors times 3 (for x, x^2, x^3)


phi_train = np.zeros((y_train.shape[0] - nfir, reg_dim))
for idx in range(nfir, N_train):
    # regressors
    regs = np.flip(u_train[idx - nfir : idx + 1])
    # nonlinear expansion of regressors
    regs_exp = np.concatenate(
        [np.power(regs, i) for i in range(1, max_power + 1)]
    )  # x, x^2, x^3
    phi_train[idx - nfir, :] = regs_exp

phi_test = np.zeros((y_test.shape[0] - nfir, reg_dim))
for idx in range(nfir, N_test):
    regs = np.flip(u_test[idx - nfir : idx + 1])
    regs_exp = np.concatenate(
        [np.power(regs, i) for i in range(1, max_power + 1)]
    )  # x, x^2, x^3
    phi_test[idx - nfir, :] = regs_exp

# Align output
y_train_nfir = np.reshape(y_train[nfir:], (-1, 1))
y_test_nfir = np.reshape(y_test[nfir:], (-1, 1))

phi_train.shape, y_train_nfir.shape

### FIR training without L1 regularization 

Least Squares implementation in PyTorch with gradient descent. We may use any other linear solver instead!

In [ ]:
# FIR identification (without LASSO)

phi_train_t = torch.tensor(phi_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_nfir, dtype=torch.float32)
theta = torch.randn((reg_dim, 1), dtype=torch.float32, requires_grad=True)


def MSE_loss(theta, reg_train_t, y_train_t):
    err = torch.matmul(reg_train_t, theta) - y_train_t
    loss = torch.mean(err**2)
    return loss


loss = MSE_loss(theta, phi_train_t, y_train_t)
loss

In [ ]:
max_iter = 100_000
lr = 1e-3

optimizer = torch.optim.SGD([theta], lr=lr)

for train_iter in range(max_iter):

    optimizer.zero_grad()

    # forward pass
    loss = MSE_loss(theta, phi_train_t, y_train_t)

    # backward pass
    loss.backward()

    optimizer.step()

    if train_iter % 10_000 == 0:
        print(f"Iteration: {train_iter}, loss: {loss:2f}")

In [ ]:
# theta_fir_ls = np.linalg.lstsq(reg_train_t, y_train_t, rcond=None)[0]

In [ ]:
# simulation performance
def simulate_fir(theta, u_sim, nfir, y0=None):

    N = u_sim.shape[0]
    y_sim = np.zeros(N)
    if y0 is not None:
        y_sim[0:nfir] = y0

    for idx in range(nfir, N):
        x_reg_sim = np.flip(u_sim[idx - nfir : idx + 1])
        y_sim[idx] = np.matmul(x_reg_sim, theta)[0]

    return y_sim

In [ ]:
y_train_sim = simulate_fir(theta.detach().numpy(), u_train, nfir)
mse_train, R2_train = regression_metrics(y_train_nfir, y_train_sim[nfir:])
print(f"Train - MSE: {mse_train:.4f}, R2_train: {R2_train:.4f}")


y_test_sim = simulate_fir(theta.detach().numpy(), u_test, nfir, y0=np.zeros(nfir))
mse_test, R2_test = regression_metrics(y_test_nfir, y_test_sim[nfir:])
print(f"Test - MSE: {mse_test:.4f}, R2_test: {R2_test:.4f}")

plt.figure()
plt.plot(y_test_nfir * y_std + y_mean, label="True"),
plt.plot(y_test_sim[nfir:] * y_std + y_mean, label="Simulated")
plt.title("Test data")
plt.legend()
plt.show()

### FIR training with L1 regularization 

Implementation in PyTorch with proximal gradient descent. We may use some specialized LASSO solver instead... 

Let us first implement soft thresholding, which is the proximal operator for the 1-norm, as a PyTorch function

In [ ]:
def soft_thresholding(theta, th):
    # theta: tensor of parameters
    # th: threshold (scalar or tensor broadcastable to theta)
    return torch.sign(theta) * torch.maximum(
        torch.abs(theta) - th, torch.zeros_like(theta)
    )


theta_th = soft_thresholding(theta, th=1.0)
torch.concatenate((theta, theta_th), dim=1)

Proximal Gradient Method for LASSO

In [ ]:
# FIR identification with LASSO

theta = torch.randn((reg_dim, 1), dtype=torch.float32, requires_grad=True)

max_iter = 100_000
lr = 1e-3
lambda_l1 = 0.2  # l1 regularization strength

optimizer = torch.optim.SGD([theta], lr=lr)

for train_iter in range(max_iter):

    optimizer.zero_grad()

    # forward pass (loss computation)
    loss = MSE_loss(theta, phi_train_t, y_train_t)

    # backward pass (loss gradient computation)
    loss.backward()

    optimizer.step()

    with torch.no_grad():
        theta.data = soft_thresholding(theta.data, th=lambda_l1 * lr)

    if train_iter % 1000 == 0:
        print(f"Iteration: {train_iter}, loss: {loss:2f}")

In [ ]:
print(f"Estimated parameters: {theta}")

In [ ]:
y_train_sim = simulate_fir(theta.detach().numpy(), u_train, nfir)
mse_train, R2_train = regression_metrics(y_train[nfir:], y_train_sim[nfir:])
print(f"Train - MSE: {mse_train:.4f}, R2_train: {R2_train:.4f}")


y_test_sim = simulate_fir(theta.detach().numpy(), u_test, nfir, y0=np.zeros(nfir))
mse_test, R2_test = regression_metrics(y_test[nfir:], y_test_sim[nfir:])
print(f"Test - MSE: {mse_test:.4f}, R2_test: {R2_test:.4f}")

plt.figure()
plt.plot(y_test[nfir:] * y_std + y_mean, label="True"),
plt.plot(y_test_sim[nfir:] * y_std + y_mean, label="Simulated")
plt.title("Test data")
plt.legend()
plt.show()

### Re-estimation only with regressors selected by L1-regularization

In [ ]:
theta_nonzero = torch.abs(theta.data) > 1e-6

reg_train_sparse = phi_train[:, theta_nonzero.squeeze()]
reg_test_sparse = phi_test[:, theta_nonzero.squeeze()]


theta_sparse = np.linalg.solve(
    np.matmul(reg_train_sparse.transpose(), reg_train_sparse),
    np.matmul(reg_train_sparse.transpose(), y_train_nfir),
)
print(theta_sparse.shape)
print(f"Least-square estimate: \n {theta_sparse}")

In [ ]:
y_train_sim = np.matmul(reg_train_sparse, theta_sparse)
mse_train, R2_train = regression_metrics(y_train_nfir, y_train_sim)
print(f"Train - MSE: {mse_train:.4f}, R2_train: {R2_train:.4f}")


y_test_sim = np.matmul(reg_test_sparse, theta_sparse)
mse_test, R2_test = regression_metrics(y_test_nfir, y_test_sim)
print(f"Test - MSE: {mse_test:.4f}, R2_test: {R2_test:.4f}")

plt.figure()
plt.plot(y_test_nfir * y_std + y_mean, label="True"),
plt.plot(y_test_sim * y_std + y_mean, label="Simulated")
plt.title("Test data")
plt.legend()
plt.show()